# Pipeline d'Extraction KG avec Ontologie
**Pipeline : OwlReady2 → Pydantic → LLM (Instructor/Groq)**

* **OwlReady2 :** permet de charger et manipuler une ontologie pour structurer les concepts et relations métier..
* **Pydantic :** sert à définir des schémas de données stricts pour valider et typer les entités extraites.
* **Instructor:** guide le LLM pour produire des sorties conformes aux modèles Pydantic de manière fiable.

* **FOAF :** (Friend of a Friend) : une ontologie du web sémantique qui permet de décrire des personnes, leurs informations de profil et leurs relations sociales de manière standardisée..

## 0. Installation

In [ ]:
!pip install -q owlready2 pydantic instructor groq pymupdf pymupdf4llm
print("Installation terminée")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 43.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 108.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 109.1 MB/s eta 0:00:00
Installation terminée


## 1. Imports & Configuration

In [ ]:
from owlready2 import *
from pydantic import BaseModel, Field, model_validator
from typing import List, Optional
import instructor
from enum import Enum
import os

from google.colab import userdata
from groq import Groq

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


## 2. Chargement de l'Ontologie

In [ ]:
!wget -q -O foaf.rdf "http://xmlns.com/foaf/spec/index.rdf"

onto = get_ontology("file://foaf.rdf").load()
print("Ontologie chargée")

Ontologie chargée


* Owlready2 * WARNING: DataProperty http://xmlns.com/foaf/0.1/name belongs to more than one entity types: [owl.DatatypeProperty, rdf-schema.label]; I'm trying to fix it...


## 3. Extraction du Vocabulaire OWL

On extrait classes, propriétés, les restrictions, domain/range, hiérarchie.

In [ ]:
classes_list    = [cls.name for cls in onto.classes() if cls.name]
relations_list  = [prop.name for prop in onto.object_properties() if prop.name]
attributs_list  = [att.name for att in onto.data_properties() if att.name]

print(f"-> {len(classes_list)} classes   : {classes_list[:10]}")
print(f"-> {len(relations_list)} relations : {relations_list[:10]}")
print(f"-> {len(attributs_list)} attributs : {attributs_list[:10]}")

-> 15 classes   : ['Class', 'LabelProperty', 'Person', 'Agent', 'SpatialThing', 'Organization', 'Project', 'Document', 'Group', 'Image']
-> 33 relations : ['mbox', 'based_near', 'phone', 'homepage', 'page', 'weblog', 'openid', 'tipjar', 'made', 'maker']
-> 27 attributs : ['mbox_sha1sum', 'gender', 'geekcode', 'dnaChecksum', 'sha1', 'title', 'nick', 'jabberID', 'aimChatID', 'skypeID']


### Restrictions OWL :
1. SOME (∃ → "il existe au moins un") : au moins une relation vers un élément de ce type

2. ONLY (∀ → "seulement") : toutes les valeurs doivent être de ce type

3. VALUE (hasValue) : valeur fixe imposée

4. MIN (≥) : au moins N relations

5. MAX (≤) : au plus N relations

6. EXACTLY (=) : exactement N relations


exemple :  
r	r.property	r.value	r.cardinality	r.type \
r1	   aEnfant	Personne	None	SOME

In [ ]:

def restriction_vers_texte(restriction, nom_classe):
    """
    Convertit un objet owlready2.Restriction en phrase
    compréhensible pour un LLM.
    Gère : SOME, ONLY, VALUE, MIN, MAX, EXACTLY
    """
    prop      = getattr(restriction, 'property', None) # property (relation) on which the restriction is applied

    valeur    = getattr(restriction, 'value', None)
    cardina   = getattr(restriction, 'cardinality', None)

    prop_nom   = prop.name   if prop                   else "?"
    valeur_nom = valeur.name if hasattr(valeur, 'name') else str(valeur)

    mapping = {
        SOME:    f"A '{nom_classe}' must have AT LEAST ONE '{prop_nom}' of type '{valeur_nom}'.",
        ONLY:    f"A '{nom_classe}' can ONLY have '{prop_nom}' of type '{valeur_nom}'.",
        VALUE:   f"A '{nom_classe}' strictly has the fixed value '{valeur_nom}' for '{prop_nom}'.",
        MIN:     f"A '{nom_classe}' must have a MINIMUM of {cardina} '{prop_nom}'.",
        MAX:     f"A '{nom_classe}' must have a MAXIMUM of {cardina} '{prop_nom}'.",
        EXACTLY: f"A '{nom_classe}' must have EXACTLY {cardina} '{prop_nom}'.",

    }
    return mapping.get(restriction.type,
                       f"Restriction on '{prop_nom}' for '{nom_classe}': {str(restriction)}")


def build_regles_restrictions(onto):
    """Retourne un bloc texte listant toutes les restrictions de l'ontologie."""
    lignes = []
    for classe in onto.classes():
        conditions = list(classe.is_a) + list(classe.equivalent_to)
        for c in conditions:
            if isinstance(c, Restriction):
                phrase = restriction_vers_texte(c, classe.name)
                lignes.append(f"  - {phrase}")
    return "\n".join(lignes) if lignes else ""


# Domain / Range des propriétés
#domain = quels types de classes peuvent être l’objet source
#range = quels types de classes peuvent être la cible

def build_domain_range(onto):
    """
    Indique au LLM quelles entités peuvent posséder quels attributs,
    et surtout quel est le type de donnée attendu (int, str, date...).
    """
    lignes = []
    for prop in onto.object_properties():
        domains = [d.name for d in prop.domain if hasattr(d, 'name')]
        ranges  = [r.name for r in prop.range  if hasattr(r, 'name')]
        if domains or ranges:
            src = ' or '.join(domains) if domains else 'any entity'
            tgt = ' or '.join(ranges)  if ranges  else 'any entity'
            lignes.append(f"  - '{prop.name}' : Source is {src} and Target is {tgt}")
    return "\n".join(lignes) if lignes else "  (no source/target declared)"

def build_attributs_domain_range(onto):
    """
    Indicates to the LLM which entities can possess which attributes,
    and especially what data type is expected (int, str, date...).
    """
    lignes = []

    # Loop over DataProperties (attributes)
    for prop in onto.data_properties():
        # The Domain (who can possess this attribute?)
        domains = [d.name for d in prop.domain if hasattr(d, 'name')]

        # The Range (what type of value? int, str, float...)
        ranges = [r.__name__ for r in prop.range if hasattr(r, '__name__')]

        if domains or ranges:
            src = ' or '.join(domains) if domains else 'any entity'
            tgt = ' or '.join(ranges)  if ranges  else 'str (default text)'

            lignes.append(f"  - '{prop.name}' : applicable to [{src}], expected format: [{tgt}]")

    return "\n".join(lignes) if lignes else " "


# Hiérarchie de classes

def build_hierarchie(onto):
    """
    Lists the subtyping relations to guide the LLM
    in choosing the correct entity type.
    """
    lignes = []
    for cls in onto.classes():
        parents = [
            p.name for p in cls.is_a
            if isinstance(p, ThingClass) and p.name != cls.name
        ]
        if parents:
            lignes.append(f"  - '{cls.name}' is a subtype of: {parents}")
    return "\n".join(lignes) if lignes else "  (flat hierarchy)"


# Propriétés logiques (symétrique, transitif, etc.)

regles_owl_vers_texte = {
    SymmetricProperty:         "This relation works in both directions (if A relates to B, then B relates to A).",
    AsymmetricProperty:        "This relation is strictly one-way.",
    TransitiveProperty:        "This relation is transitive (if A→B and B→C, then A→C).",
    FunctionalProperty:        "A source entity can have ONLY ONE target for this relation.",
    InverseFunctionalProperty: "The target acts as a unique identifier (e.g., email, social security number).",
    ReflexiveProperty:         "An entity always has this relation with itself.",
    IrreflexiveProperty:       "An entity can NEVER have this relation with itself."
}

def build_proprietes_logiques(onto):
    lignes = []
    for prop in onto.object_properties():
        regles = [
            phrase for type_owl, phrase in regles_owl_vers_texte.items()
            if type_owl in prop.is_a
        ]
        if regles:
            lignes.append(f"  - '{prop.name}' : {' '.join(regles)}")
    return "\n".join(lignes) if lignes else " "


# Génération de tous les blocs
bloc_restrictions   = build_regles_restrictions(onto)
bloc_domain_range   = build_domain_range(onto)
bloc_attributs_range = build_attributs_domain_range(onto)
bloc_hierarchie     = build_hierarchie(onto)
bloc_logique        = build_proprietes_logiques(onto)

# --- GÉNÉRATION DES MAPPINGS DE RÈGLES ---

domain_map = {
    prop.name: [d.name for d in prop.domain if hasattr(d, 'name')]
    for prop in onto.object_properties()
}

range_map = {
    prop.name: [r.name for r in prop.range if hasattr(r, 'name')]
    for prop in onto.object_properties()
}

print(f"Mappings générés : {len(domain_map)} domaines et {len(range_map)} ranges indexés.")



print("=== RESTRICTIONS ===")
print(bloc_restrictions[:500], "...\n")
print("=== DOMAIN / RANGE ===")
print(bloc_domain_range[:500], "...\n")
print("=== ATTRIBUTS / RANGE ===")
print(bloc_attributs_range[:500], "...\n")
print("=== HIÉRARCHIE ===")
print(bloc_hierarchie[:500])

Mappings générés : 33 domaines et 33 ranges indexés.
=== RESTRICTIONS ===
 ...

=== DOMAIN / RANGE ===
  - 'mbox' : Source is Agent and Target is Thing
  - 'based_near' : Source is SpatialThing and Target is SpatialThing
  - 'homepage' : Source is Thing and Target is Document
  - 'page' : Source is Thing and Target is Document
  - 'weblog' : Source is Agent and Target is Document
  - 'openid' : Source is Agent and Target is Document
  - 'tipjar' : Source is Agent and Target is Document
  - 'made' : Source is Agent and Target is Thing
  - 'maker' : Source is Thing and Target is Agent
  - 'img' : S ...

=== ATTRIBUTS / RANGE ===
  - 'mbox_sha1sum' : applicable to [Agent], expected format: [str (default text)]
  - 'gender' : applicable to [Agent], expected format: [str (default text)]
  - 'geekcode' : applicable to [Person], expected format: [str (default text)]
  - 'sha1' : applicable to [Document], expected format: [str (default text)]
  - 'jabberID' : applicable to [Agent], expected fo

## 4. Schémas Pydantic

In [ ]:
# Enums verrouillés sur le vocabulaire de l'ontologie
VocabulaireEntites   = Enum('VocabulaireEntites',   {m: m for m in classes_list})
VocabulaireRelations = Enum('VocabulaireRelations',  {m: m for m in relations_list})
VocabulaireAttributs = Enum('VocabulaireAttributs',  {m: m for m in attributs_list})


class AttributExtrait(BaseModel):
    cle:    VocabulaireAttributs = Field(description="Official name of the attribute (e.g., 'age', 'name')")
    valeur: str                  = Field(description="Value found in the text")

class Entite(BaseModel):
    id_entite: str = Field(description="UNIQUE ID. You MUST use the key 'id_entite', NOT 'id'.")
    label: str = Field(description="The display name. You MUST use the key 'label'.")
    type_entite: VocabulaireEntites = Field(description="Ontology type. You MUST use the key 'type_entite', NOT 'type'.")
    attributs:    List[AttributExtrait] = Field(default=[], description="Characteristics of the entity")

class Relation(BaseModel):
    source_id:     str                  = Field(description="source entity ID")
    type_relation: VocabulaireRelations = Field(description="Relationship type according to the ontology")
    cible_id:      str                  = Field(description="target entity ID")

class GrapheExtrait(BaseModel):
    entites:   List[Entite]
    relations: List[Relation]

    @model_validator(mode='after')
    def verifier_coherence_relations(self):
        # Vérification que les IDs source/cible existent bien dans la liste des entités
        ids_valides = {e.id_entite for e in self.entites}
        avant = len(self.relations)
        self.relations = [
            r for r in self.relations
            if r.source_id in ids_valides and r.cible_id in ids_valides
        ]
        if avant != len(self.relations):
            print(f"  [Validation] {avant - len(self.relations)} relation(s) orpheline(s) supprimée(s).")
        return self

print("Schémas Pydantic prêts")

Schémas Pydantic prêts


## 5. Construction du Prompt Système

In [ ]:
# prompt_systeme = f"""
# # OBJECTIVE
# You are a Semantic Web expert. Analyze the provided text and extract a knowledge graph
# structured according to the ontology below. STRICTLY respect the authorized vocabulary.
# NEVER TRY TO CREATE AN ENTITIES, ATTRIBUTES, RELATIONSHIPS THAT IS NOT AUTHORIZED BELLOW
# JUST SKIP IT IF IT NOT ALLOWED or if You don't Know

# # ANALYSIS STEPS
# 1. ENTITIES    : Identify each relevant entity and determine its exact type.
# 2. ATTRIBUTES  : For each entity, extract its direct characteristics (name, age, etc.).
# 3. RELATIONSHIPS: Identify the links between entities. Respect the direction of the arrow (Source → Target).

# # AUTHORIZED VOCABULARY (use these terms exactly as written, do not translate them)
# * ENTITY TYPES       : {classes_list}
# * RELATIONSHIP TYPES : {relations_list}
# * ATTRIBUTE TYPES    : {attributs_list}

# # CLASS HIERARCHY
# Always prefer the most specific type:
# {bloc_hierarchie}

# # DOMAIN / RANGE CONSTRAINTS
# For each relationship, verify that the source and target are of the correct type:
# {bloc_domain_range}


# # CONTRAINTES DES ATTRIBUTS (DataProperties)
# Ensure that an attribute is applied only to authorized entities and adheres to the value format (e.g., if the type is ‘int’, enter a number):
# {bloc_attributs_range}

# # STRUCTURAL RESTRICTIONS
# These constraints define what an entity MUST or CAN have:
# {bloc_restrictions}

# # LOGICAL PROPERTIES OF RELATIONSHIPS
# {bloc_logique}

# # GENERAL RULES
# - Deduplication: if a person is mentioned multiple times, merge them under a single id_entite.
# - id_entite: use a stable snake_case format.
# - Do not invent any information that is absent from the text.
# - Check for each realtion that the source entity and the target entity are allowed throught Domain/Range constraints.
# - If any information is uncertain, omit it rather than inventing it.


# Return ONLY the raw JSON. No conversational text, no Markdown blocks. If you use Markdown, the system will crash.
# """

# print("System prompt generated")
# print(f"Length: {len(prompt_systeme)} characters")

In [ ]:
prompt_systeme = f"""You are a top-tier algorithm designed for extracting information in JSON structured formats to build a knowledge graph.

### Output Format
Generate a knowledge graph in JSON format using the following structure:

**Entity:**
- **id_entite:** A unique identifier (string). Use a human-readable identifier as found in the text (e.g., "").
- **label:** The display name (string), must be one of the allowed entity types (e.g., "Person").
- **type_entite:** The type of the entity (string), must be one of the allowed entity types (e.g., "Person").
- **attributs:** For each entity, extract its direct characteristics (name, age, etc.).

**Relationships:**
- **source_id:** The id of the source entity.
- **type_relation:** The type of the relationship, must be one of the allowed relationship types (e.g., made).
- **cible_id:** The id of the target entity.

### Schema Definition (STRICT)
The schema below must be followed exactly. Only use the allowed types, properties, and relationship types listed, and do not introduce any additional elements.

**Allowed entity Types:**
{classes_list}

**Allowed Relationship:**
{relations_list}

**Allowed entity Attributes:**
{attributs_list}

**STRUCTURAL RESTRICTIONS:**
These constraints define what an entity MUST or CAN have:
{bloc_restrictions}

**CLASS HIERARCHY:**
Always prefer the most specific type:
{bloc_hierarchie}

**DOMAIN / RANGE CONSTRAINTS:**
For each relationship, verify that the source and target are of the correct type:
{bloc_domain_range}

**LOGICAL PROPERTIES OF RELATIONSHIPS:**
{bloc_logique}

**Attribute RULES:**
Ensure that an attribute is applied only to authorized entities and adheres to the value format (e.g., if the type is ‘int’, enter a number):
{bloc_attributs_range}

### Rules to Follow:
- Do not add any information that is not explicitly mentioned in the text.
- Relationship source and target must reference valid node ids.
- Omit any empty fields from the output.

### Task:
Output only a valid JSON object.
"""

print("System prompt generated")
print(f"Length: {len(prompt_systeme)} characters")


System prompt generated
Length: 7635 characters


In [ ]:
print(prompt_systeme)

You are a top-tier algorithm designed for extracting information in JSON structured formats to build a knowledge graph.

### Output Format
Generate a knowledge graph in JSON format using the following structure:

**Entity:**
- **id_entite:** A unique identifier (string). Use a human-readable identifier as found in the text (e.g., "").
- **label:** The display name (string), must be one of the allowed entity types (e.g., "Person").
- **type_entite:** The type of the entity (string), must be one of the allowed entity types (e.g., "Person").
- **attributs:** For each entity, extract its direct characteristics (name, age, etc.).

**Relationships:**
- **source_id:** The id of the source entity.
- **type_relation:** The type of the relationship, must be one of the allowed relationship types (e.g., made).
- **cible_id:** The id of the target entity.

### Schema Definition (STRICT)
The schema below must be followed exactly. Only use the allowed types, properties, and relationship types listed,

## 6. Client LLM

In [ ]:
client = instructor.from_provider("groq/llama-3.3-70b-versatile")
print("Client Instructor/Groq prêt ")

Client Instructor/Groq prêt 


## 7. Fonction d'Extraction

In [ ]:
def extraire_graphe(texte: str) -> GrapheExtrait:
    """Extrait un graphe avec gestion d'erreurs."""
    try:
        return client.create(
            response_model=GrapheExtrait,
            messages=[
                {"role": "system", "content": prompt_systeme},
                {"role": "user",   "content": texte}
            ]
        )
    except Exception as e:
        print(f"Échec de l'extraction: {e}")
        return GrapheExtrait(entites=[], relations=[])

print("Fonction extraire_graphe() prête")

Fonction extraire_graphe() prête


## 8. Extraction avec Chunking (pour les longs textes / PDFs)

Découpe le texte en morceaux, extrait un graphe par chunk,
puis fusionne en dédoublonnant sur `id_entite`.

In [ ]:
def extraire_graphe_chunked(
    texte: str,
    chunk_size: int = 3000,
    overlap: int = 200
) -> GrapheExtrait:
    """
    Extrait un graphe depuis un long texte en le découpant en chunks.
    Le recouvrement (overlap) permet de ne pas perdre les entités
    mentionnées en fin/début de chunk.

    Args:
        texte      : texte brut à analyser
        chunk_size : taille max d'un chunk en caractères
        overlap    : recouvrement entre chunks consécutifs
    """
    # 1. Découpage avec recouvrement
    chunks = []
    start  = 0
    while start < len(texte):
        end = start + chunk_size
        chunks.append(texte[start:end])
        start = end - overlap  # recouvrement

    print(f"  {len(chunks)} chunk(s) à traiter...")

    # 2. Extraction par chunk
    tous_graphes = []
    for i, chunk in enumerate(chunks):
        print(f"  Chunk {i+1}/{len(chunks)} ({len(chunk)} car.)...", end=" ")
        try:
            g = extraire_graphe(chunk)
            tous_graphes.append(g)
            print(f"{len(g.entites)} entités, {len(g.relations)} relations")
        except Exception as e:
            print(f"ERREUR : {e}")

    # 3. Fusion avec dédoublonnage sur id_entite
    entites_map = {}  # id_entite → Entite (le premier gagne)
    toutes_relations = []

    for g in tous_graphes:
        for e in g.entites:
            if e.id_entite not in entites_map:
                entites_map[e.id_entite] = e
            else:
                # Fusion des attributs manquants
                cles_existantes = {a.cle for a in entites_map[e.id_entite].attributs}
                for attr in e.attributs:
                    if attr.cle not in cles_existantes:
                        entites_map[e.id_entite].attributs.append(attr)
        toutes_relations.extend(g.relations)

    # Dédoublonnage des relations (source, type, cible)
    rels_uniques = list({(r.source_id, r.type_relation, r.cible_id): r
                         for r in toutes_relations}.values())

    graphe_final = GrapheExtrait(
        entites=list(entites_map.values()),
        relations=rels_uniques
    )
    print(f"\n  Graphe fusionné : {len(graphe_final.entites)} entités, {len(graphe_final.relations)} relations")
    return graphe_final

print("Fonction extraire_graphe_chunked() prête")

Fonction extraire_graphe_chunked() prête


## 9. Tests

### 9.1 Texte court

In [ ]:
texte_test = """

Vincent van Gogh (prononcé en néerlandais : /ˈvɪnsɛnt vɑŋ ˈɣɔx/)[Note 1], né le 30 mars 1853 à Zundert aux Pays-Bas et mort le 29 juillet 1890 à Auvers-sur-Oise en France à l'âge de 37 ans, est un peintre et dessinateur néerlandais. Son œuvre pleine de naturalisme, inspirée par l'impressionnisme et le pointillisme, annonce le fauvisme et l'expressionnisme.

Van Gogh grandit au sein d'une famille de l'ancienne bourgeoisie. Il tente d'abord de faire carrière comme marchand d'art chez Goupil & Cie. Cependant, refusant de voir l'art comme une marchandise, il est licencié. Il aspire alors à devenir pasteur, mais il échoue aux examens de théologie. À l'approche de 1880, il se tourne vers la peinture. Pendant ces années, il quitte les Pays-Bas pour la Belgique, puis s'établit en France. Vincent explore la peinture et le dessin à la fois en autodidacte et en suivant des cours. Passionné, il ne cesse d'enrichir sa culture picturale : il analyse le travail des peintres de l'époque, il visite les musées et les galeries d'art, il échange des idées avec ses amis peintres, il étudie notamment les estampes japonaises, les gravures anglaises, etc. Sa peinture reflète ses recherches et l'étendue de ses connaissances artistiques. Toutefois, sa vie est parsemée de crises qui révèlent son instabilité mentale. L'une d'elles provoque son suicide, à l'âge de 37 ans.

"""

print("--- Extraction en cours ---")
graphe = extraire_graphe(texte_test)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "VincentVanGogh",
      "label": "Vincent van Gogh",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "Vincent van Gogh"
        },
        {
          "cle": "birthday",
          "valeur": "30 mars 1853"
        },
        {
          "cle": "age",
          "valeur": "37 ans"
        }
      ]
    },
    {
      "id_entite": "PaysBas",
      "label": "Pays-Bas",
      "type_entite": "SpatialThing",
      "attributs": []
    },
    {
      "id_entite": "France",
      "label": "France",
      "type_entite": "SpatialThing",
      "attributs": []
    },
    {
      "id_entite": "Goupil&Cie",
      "label": "Goupil & Cie",
      "type_entite": "Organization",
      "attributs": []
    },
    {
      "id_entite": "Belgique",
      "label": "Belgique",
      "type_entite": "SpatialThing",
      "attributs": []
    }
  ],
  "relations": [
    {
 

In [ ]:
texte_test = """
Marc a un compte Facebook et a connu laurent sur son compte Instagram
"""

print("--- Extraction en cours ---")
graphe = extraire_graphe(texte_test)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))


--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "Marc",
      "label": "Marc",
      "type_entite": "Person",
      "attributs": []
    },
    {
      "id_entite": "Laurent",
      "label": "Laurent",
      "type_entite": "Person",
      "attributs": []
    },
    {
      "id_entite": "CompteFacebook",
      "label": "Compte Facebook",
      "type_entite": "OnlineAccount",
      "attributs": []
    },
    {
      "id_entite": "CompteInstagram",
      "label": "Compte Instagram",
      "type_entite": "OnlineAccount",
      "attributs": []
    }
  ],
  "relations": [
    {
      "source_id": "Marc",
      "type_relation": "account",
      "cible_id": "CompteFacebook"
    },
    {
      "source_id": "Marc",
      "type_relation": "account",
      "cible_id": "CompteInstagram"
    },
    {
      "source_id": "Marc",
      "type_relation": "knows",
      "cible_id": "Laurent"
    },
    {
      "source_id": "Laurent",
      "type_relation": "account",

### 9.2 PDF (avec chunking automatique)

In [ ]:
# import pymupdf4llm

# PDF_PATH = "Falling_in_Reverse-1-2.pdf"

# print(f"Lecture du PDF : {PDF_PATH}")
# texte_pdf = pymupdf4llm.to_markdown(PDF_PATH)
# print(f"{len(texte_pdf)} caractères extraits.\n")

# print("--- Extraction avec chunking ---")
# graphe_pdf = extraire_graphe_chunked(texte_pdf, chunk_size=3000, overlap=200)

# print("\n--- RÉSULTAT ---")
# print(graphe_pdf.model_dump_json(indent=2))

### 9.3 Test

In [ ]:
texte1 = """
Vincent van Gogh (1853–1890) est un peintre néerlandais majeur du post‑impressionnisme. Autodidacte en grande partie, il a produit plus de 2 000 œuvres, dont Les Tournesols, La Nuit étoilée et La Chambre à Arles. Sa vie est marquée par des difficultés psychologiques et une reconnaissance artistique tardive. Sa correspondance avec son frère Théo constitue une source biographique essentielle.
"""

In [ ]:
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte1)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "vincent_van_gogh",
      "label": "Vincent van Gogh",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "Vincent van Gogh"
        },
        {
          "cle": "birthday",
          "valeur": "1853"
        },
        {
          "cle": "age",
          "valeur": "37"
        }
      ]
    },
    {
      "id_entite": "theo_van_gogh",
      "label": "Théo van Gogh",
      "type_entite": "Person",
      "attributs": []
    },
    {
      "id_entite": "les_tournesols",
      "label": "Les Tournesols",
      "type_entite": "Image",
      "attributs": []
    },
    {
      "id_entite": "la_nuit_etoilee",
      "label": "La Nuit étoilée",
      "type_entite": "Image",
      "attributs": []
    },
    {
      "id_entite": "la_chambre_a_arles",
      "label": "La Chambre à Arles",
      "type_entite": "Image",
      "attributs": []
    }
  ],
  "rel

In [ ]:
#texte soutenu

texte = """
Le projet 'OntoGraph', dont l'édification fut magistralement orchestrée par Bob Martin, constitue l'une des fiertés de son créateur. Il est à noter qu'une connaissance mutuelle particulièrement prononcée lie ce dernier à Alice Dubois, une experte en ingénierie logicielle actuellement âgée de vingt-huit ans.
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "OntoGraph",
      "label": "Projet OntoGraph",
      "type_entite": "Project",
      "attributs": [
        {
          "cle": "name",
          "valeur": "OntoGraph"
        }
      ]
    },
    {
      "id_entite": "BobMartin",
      "label": "Bob Martin",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "Bob Martin"
        }
      ]
    },
    {
      "id_entite": "AliceDubois",
      "label": "Alice Dubois",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "Alice Dubois"
        },
        {
          "cle": "age",
          "valeur": "28"
        }
      ]
    }
  ],
  "relations": [
    {
      "source_id": "BobMartin",
      "type_relation": "made",
      "cible_id": "OntoGraph"
    },
    {
      "source_id": "BobMartin",
      "type_relation": "knows",
      "cible_id": "Ali

In [ ]:
texte = """
Moi c Alice 28 piges & dev backend !! Je taff svt ac @Bob_Martin (le big boss ki a codé le projet OntoGraph 🔥). Bref on gère.
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "Alice",
      "label": "Alice",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "age",
          "valeur": "28"
        },
        {
          "cle": "name",
          "valeur": "Alice"
        }
      ]
    },
    {
      "id_entite": "Bob_Martin",
      "label": "Bob Martin",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "Bob Martin"
        }
      ]
    },
    {
      "id_entite": "OntoGraph",
      "label": "OntoGraph",
      "type_entite": "Project",
      "attributs": []
    }
  ],
  "relations": [
    {
      "source_id": "Alice",
      "type_relation": "knows",
      "cible_id": "Bob_Martin"
    },
    {
      "source_id": "Bob_Martin",
      "type_relation": "made",
      "cible_id": "OntoGraph"
    },
    {
      "source_id": "Alice",
      "type_relation": "currentProject",
      "cible_id": "Ont

In [ ]:
#texte qui vient des social

texte = """
 Will Smith n’a jamais réussi à rendre sa femme Jada Pinkett Smith heureuse.

Jada Pinkett Smith fait des révélations choc sur sa relation avec Will Smith : malgré tous ses efforts, elle affirme n’avoir jamais été réellement heureuse dans leur couple… au point de chercher ailleurs ce qu’elle ne trouvait pas.

« J’ai réalisé que ce n’était pas sa responsabilité de me rendre heureuse. Il ne peut pas. C’est impossible. »

Elle explique avoir longtemps cru à une vision idéalisée de l’amour, pensant avoir “tout fait comme il fallait”… sans jamais se sentir épanouie.

De son côté, Will assumait une autre vision :
👉 chacun doit se rendre heureux soi-même.

Une prise de conscience tardive, qui a fini par briser les illusions… et redéfinir leur relation.
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "Will_Smith",
      "label": "Will Smith",
      "type_entite": "Person",
      "attributs": []
    },
    {
      "id_entite": "Jada_Pinkett_Smith",
      "label": "Jada Pinkett Smith",
      "type_entite": "Person",
      "attributs": []
    }
  ],
  "relations": [
    {
      "source_id": "Will_Smith",
      "type_relation": "knows",
      "cible_id": "Jada_Pinkett_Smith"
    },
    {
      "source_id": "Jada_Pinkett_Smith",
      "type_relation": "knows",
      "cible_id": "Will_Smith"
    }
  ]
}


In [ ]:
# contradiction
texte = """
Bien que beaucoup de gens fassent l'amalgame sur les forums, la développeuse Alice Dubois ne connaît absolument pas le créateur du projet OntoGraph. Elle n'a jamais croisé Bob Martin de toute sa vie
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "Alice_Dubois",
      "label": "Alice Dubois",
      "type_entite": "Person",
      "attributs": []
    },
    {
      "id_entite": "Bob_Martin",
      "label": "Bob Martin",
      "type_entite": "Person",
      "attributs": []
    }
  ],
  "relations": [
    {
      "source_id": "Alice_Dubois",
      "type_relation": "knows",
      "cible_id": "Bob_Martin"
    }
  ]
}


In [ ]:
texte = """
La galaxie d'Andromède se situe à environ 2,5 millions d'années-lumière de la Voie lactée. Son trou noir supermassif central absorbe en permanence des nuages de gaz cosmique et de poussière stellaire. Les observations télescopiques montrent une collision inéluctable entre ces deux vastes formations dans environ 4 milliards d'années, un événement cosmique titanesque qui donnera naissance à une toute nouvelle galaxie elliptique
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

In [ ]:
texte = """
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "JohnDoe",
      "label": "John Doe",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "John Doe"
        },
        {
          "cle": "age",
          "valeur": "30"
        }
      ]
    },
    {
      "id_entite": "JaneDoe",
      "label": "Jane Doe",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "Jane Doe"
        },
        {
          "cle": "age",
          "valeur": "25"
        }
      ]
    }
  ],
  "relations": [
    {
      "source_id": "JohnDoe",
      "type_relation": "knows",
      "cible_id": "JaneDoe"
    }
  ]
}


In [ ]:
texte = """
Bonjour, bla bla bla
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [],
  "relations": []
}


In [ ]:
texte = """
On a dark desert highway
Cool wind in my hair
Warm smell of colitas
Rising up through the air
Up ahead in the distance
I saw a shimmering light
My head grew heavy and my sight grew dim
I had to stop for the night
There she stood in the doorway
I heard the mission bell
And I was thinking to myself
This could be heaven or this could be hell
Then she lit up a candle
And she showed me the way
There were voices down the corridor
I thought I heard them say
Welcome to the Hotel California
Such a lovely place (such a lovely place)
Such a lovely face
Plenty of room at the Hotel California
Any time of year (any time of year)
You can find it here
Her mind is Tiffany-twisted
She got the Mercedes-Benz, uh
She got a lot of pretty, pretty boys
That she calls friends
How they danced in the courtyard
Sweet summer sweat
Some dance to remember
Some dance to forget
So I called up the captain
"Please bring me my wine"
He said, "We haven't had that spirit here since 1969"
And still those voices are calling from far away
Wake you up in the middle of the night
Just to hear them say
Welcome to the Hotel California
Such a lovely place (such a lovely place)
Such a lovely face
They're livin' it up at the Hotel California
What a nice surprise (what a nice surprise)
Bring your alibis
Mirrors on the ceiling
The pink champagne on ice
And she said, "We are all just prisoners here
Of our own device"
And in the master's chambers
They gathered for the feast
They stab it with their steely knives
But they just can't kill the beast
Last thing I remember
I was running for the door
I had to find the passage back
To the place I was before
"Relax, " said the night man
"We are programmed to receive
You can check out any time you like
But you can never leave"
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe_chunked(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---
  1 chunk(s) à traiter...
  Chunk 1/1 (1746 car.)... 4 entités, 3 relations

  Graphe fusionné : 4 entités, 3 relations

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "Hotel California",
      "label": "Hotel California",
      "type_entite": "SpatialThing",
      "attributs": []
    },
    {
      "id_entite": "woman",
      "label": "woman",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "unknown"
        }
      ]
    },
    {
      "id_entite": "narrator",
      "label": "narrator",
      "type_entite": "Person",
      "attributs": []
    },
    {
      "id_entite": "captain",
      "label": "captain",
      "type_entite": "Person",
      "attributs": []
    }
  ],
  "relations": [
    {
      "source_id": "narrator",
      "type_relation": "based_near",
      "cible_id": "Hotel California"
    },
    {
      "source_id": "woman",
      "type_relation": "knows",
      "cible_id": "n

## 10. Changer d'Ontologie


In [ ]:
import owlready2

print("Chargement de l'Ontologie Pizza depuis GitHub")

url_pizza = "https://raw.githubusercontent.com/owlcs/pizza-ontology/master/pizza.owl"

onto_pizza = owlready2.get_ontology(url_pizza).load()

print(f"Succès ! Nombre de classes chargées : {len(list(onto_pizza.classes()))}")

In [ ]:
classes_list    = [cls.name for cls in onto_pizza.classes() if cls.name]
relations_list  = [prop.name for prop in onto_pizza.object_properties() if prop.name]
attributs_list  = [att.name for att in onto_pizza.data_properties() if att.name]

print(f"-> {len(classes_list)} classes   : {classes_list[:10]}")
print(f"-> {len(relations_list)} relations : {relations_list[:10]}")
print(f"-> {len(attributs_list)} attributs : {attributs_list[:10]}")

In [ ]:
# Génération de tous les blocs
bloc_restrictions   = build_regles_restrictions(onto_pizza)
bloc_domain_range   = build_domain_range(onto_pizza)
bloc_hierarchie     = build_hierarchie(onto_pizza)
bloc_logique        = build_proprietes_logiques(onto_pizza)


print("=== RESTRICTIONS ===")
print(bloc_restrictions[:500], "...\n")
print("=== DOMAIN / RANGE ===")
print(bloc_domain_range[:500], "...\n")
print("=== HIÉRARCHIE ===")
print(bloc_hierarchie[:500])

In [ ]:
# Enums verrouillés sur le vocabulaire de l'ontologie
VocabulaireEntites   = Enum('VocabulaireEntites',   {m: m for m in classes_list})
VocabulaireRelations = Enum('VocabulaireRelations',  {m: m for m in relations_list})
VocabulaireAttributs = Enum('VocabulaireAttributs',  {m: m for m in attributs_list})


class AttributExtrait(BaseModel):
    cle:    VocabulaireAttributs = Field(description="Nom officiel de l'attribut (ex: 'age', 'name')")
    valeur: str                  = Field(description="Valeur trouvée dans le texte")


class Entite(BaseModel):
    id_entite:    str                  = Field(description="ID unique stable (ex: 'person_steve_jobs')")
    label:        str                  = Field(description="Nom tel qu'il apparaît dans le texte")
    type_entite:  VocabulaireEntites   = Field(description="Type exact selon l'ontologie")
    attributs:    List[AttributExtrait] = Field(default=[], description="Caractéristiques de l'entité")


class Relation(BaseModel):
    source_id:     str                  = Field(description="id_entite de la source")
    type_relation: VocabulaireRelations = Field(description="Type de relation selon l'ontologie")
    cible_id:      str                  = Field(description="id_entite de la cible")


class GrapheExtrait(BaseModel):
    entites:   List[Entite]
    relations: List[Relation]

    # -------------------------------------------------------
    # Supprime les relations dont source ou cible
    # ne correspondent à aucune entité extraite
    # -------------------------------------------------------
    @model_validator(mode='after')
    def verifier_coherence_relations(self):
        ids_valides = {e.id_entite for e in self.entites}
        avant = len(self.relations)
        self.relations = [
            r for r in self.relations
            if r.source_id in ids_valides and r.cible_id in ids_valides
        ]
        apres = len(self.relations)
        if avant != apres:
            print(f"  [Validation] {avant - apres} relation(s) orpheline(s) supprimée(s).")
        return self

print("Schémas Pydantic prêts")

In [ ]:
texte_test = """
La pizza Margherita est composée d'une base fine classique, recouverte de sauce tomate et de mozzarella. En revanche, la pizza Américaine utilise une base épaisse et contient de la mozzarella, de la tomate, et des rondelles de saucisse pepperoni.
"""

print("--- Extraction en cours ---")
graphe = extraire_graphe(texte_test)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

In [ ]:
texte_pdf = """
Ce plat italien qui s’est exporté à l’international séduit facilement et s’est adapté à diverses cultures culinaires. Facile à préparer, mais incroyablement délicieux, plusieurs recettes ont été élaborées à travers le monde. Il existe désormais de nombreux types de pizza : les recettes traditionnelles italiennes, en passant par les versions revisitées, jusqu’aux créations les plus récentes. Votre pizzeria à Montpellier vous invite à voyager en vous présentant les différents types de pizza au fil du temps.

D’où vient la pizza et quelle est la recette originale ?
La pizza a vu le jour à Naples, en Italie, au 18e siècle. Une Mama, préparant le dîner, a eu l’idée de réaliser une pâte, de l’enduire de sauce tomates, puis de le garnir de restes d’aliments avant de recouvrir avec du fromage râpé. Cette recette expresse et facile à réaliser s’est rapidement vulgarisée auprès des classes pauvres, puis dans les différentes régions d’Italie, avant de s’exporter à l’international. Plusieurs versions ont alors été créées en s’adaptant aux spécialités régionales et aux préférences gustatives des consommateurs.

La pizza est préparée avec trois éléments principaux : une pâte qui peut être soit fine, soit épaisse ; une sauce tomate ou une crème ou encore du pesto et une garniture (légumes, viandes, poissons, fromages, etc.). Le tout peut, ensuite, être recouvert de fromage râpé. Certaines recettes commencent par le fromage, avant de rajouter la sauce par-dessus.

Quels sont les principaux types de pizza italienne ?
L’Italie, le berceau de cette recette, conserve le titre de référence mondiale avec les recettes traditionnelles qui méritent toute notre attention. Voici les 5 variantes de base qu’il faut connaître :

La Margherita
Cette pizza napolitaine reste la plus symbolique. Baptisée d’après le nom de la reine Marguerite de Savoie, elle reprend les couleurs du drapeau italien avec la sauce tomate rouge, le basilic pour le vert et la mozzarella pour le blanc.

La pizza Marinara
Les amateurs de recettes simples apprécieront cette préparation authentique sans fromage. Il s’agit d’un classique napolitain préparé avec de la sauce tomate, de l’ail, de l’origan et de l’huile d’olive.

La pizza quatre fromages ou Quattro Formaggi
A base de sauce tomate ou de crème, cette pizza est réalisée avec 4 fromages différents qui sont généralement de la mozzarella, du parmesan, du gorgonzola et de la fontina ou provolone.

La pizza Capricciosa
Elle convient à ceux qui recherchent un mélange gourmand et savoureux : préparée avec de la tomate, de la mozzarella, des artichauts, des champignons, du jambon et des olives, la pizza capricciosa compte parmi les recettes les plus appréciées.

"""


graphe_pdf = extraire_graphe_chunked(texte_pdf, chunk_size=3000, overlap=200)

print("\n--- RÉSULTAT ---")
print(graphe_pdf.model_dump_json(indent=2))

# STOCKAGE

## Allegro

In [ ]:
!pip install -q agraph-python

Connexion au repository

In [ ]:
from franz.openrdf.connect import ag_connect
import os

AG_HOST = "ag1twvp94ekf9p0n.allegrograph.cloud"
AG_USER = "admin"
AG_PASS = "xqJ02HkQ48zph5vgej76rq"
REPO_NAME = "foaf_graph"

try:
    # Connexion au serveur et création/ouverture du repository
    conn = ag_connect(
        REPO_NAME,
        host=AG_HOST,
        port=443, # Port standard pour le cloud (HTTPS)
        user=AG_USER,
        password=AG_PASS,
        protocol='https'
    )
    print(f"Connecté à AllegroGraph ! Repository: {REPO_NAME}")
    print(f"Nombre de triplets actuels : {conn.size()}")

except Exception as e:
    print(f" Erreur de connexion : {e}")

KeyboardInterrupt: 

In [ ]:
# Définition des URIs (Sujet, Prédicat, Objet)
alice = conn.createURI("http://example.org/people/alice")
type_rel = conn.createURI("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")
person_class = conn.createURI("http://xmlns.com/foaf/0.1/Person")

# Ajout au graphe
conn.add(alice, type_rel, person_class)
conn.commit()

print(f"Succès ! Nouveau nombre de triplets : {conn.size()}")

Succès ! Nouveau nombre de triplets : 638


In [ ]:
# def envoie_graph_allegro(graphe: GrapheExtrait, conn, onto_uri):
#     """envoie le graphe extraite à Allegro"""
#     print(f"size avant ajouts: {conn.size()}")

#     BASE_URI = "http://mon_projet.org/"

#     liste_entite = graphe.entites
#     liste_relation = graphe.relations

#     #pour les entités et leur attribut
#     for i in range(len(liste_entite)):

#       entite = conn.createURI(BASE_URI + liste_entite[i].id_entite)
#       type_rel = conn.createURI(onto_uri)
#       type_class = conn.createURI(liste_entite[i].type_entite.value)
#       conn.add(entite, type_rel, type_class)
#       for j in range(len(liste_entite[i].attributs)):
#         attribut = conn.createURI(onto_uri+ liste_entite[i].attributs[j].cle)
#         attribut_value = conn.createLiteral(liste_entite[i].attributs[j].valeur)
#         conn.add(entite, attribut, attribut_value)

#       # pour les relations
#       for k in range(len(liste_relation)):

#         source = conn.createURI(BASE_URI + liste_relation[k].source_id)
#         type_rel = conn.createURI(onto_uri + liste_relation[k].type_relation.value)
#         cible = conn.createURI(BASE_URI + liste_relation[k].cible_id)
#         conn.add(source, type_rel, cible)

#       conn.commit()

#       return f"Succès ! Nouveau nombre de triplets : {conn.size()}"


#Correction :

def envoie_graph_allegro(graphe, conn, onto_uri_base):
    """Envoie le graphe extrait à AllegroGraph"""
    print(f"Taille avant ajouts: {conn.size()}")


    BASE_URI = "http://mon_projet.org/"
    # URI standard pour définir le type d'une entité
    RDF_TYPE = conn.createURI("http://www.w3.org/1999/02/22-rdf-syntax-ns#type")

    # --- ÉTAPE 1 : LES ENTITÉS ET ATTRIBUTS ---
    for entite_data in graphe.entites:
        # Création de l'URI du sujet (ex: http://mon_projet.org/Alice)
        sujet = conn.createURI(BASE_URI + entite_data.id_entite.replace(" ", "_"))

        # Définition du type (ex: foaf:Person)
        nom_classe = entite_data.type_entite.value
        classe_uri = conn.createURI(onto_uri_base + nom_classe)
        conn.add(sujet, RDF_TYPE, classe_uri)

        # Ajout des attributs (Datatype Properties)
        for attr in entite_data.attributs:
            # On construit l'URI de la propriété (ex: foaf:name)
            propriete = conn.createURI(onto_uri_base + attr.cle.value)
            valeur = conn.createLiteral(attr.valeur)
            conn.add(sujet, propriete, valeur)

    # --- ÉTAPE 2 : LES RELATIONS ---
    for rel in graphe.relations:
        source = conn.createURI(BASE_URI + rel.source_id.replace(" ", "_"))
        type_rel = conn.createURI(onto_uri_base + rel.type_relation.value)
        cible = conn.createURI(BASE_URI + rel.cible_id.replace(" ", "_"))
        conn.add(source, type_rel, cible)

    # On valide les changements
    conn.commit()

    return f"Succès ! Nouveau nombre de triplets : {conn.size()}"



### Envoie a Allegro

In [ ]:
texte = """
William Shakespeare est un dramaturge, poète et acteur anglais baptisé/né le 26 avril 1564 à Stratford-upon-Avon et mort le 23 avril 1616 dans la même ville. Surnommé « le Barde d'Avon », « le Barde immortel » ou simplement « le Barde », il est considéré comme l'un des plus grands poètes et dramaturges de langue anglaise. Son œuvre, traduite dans de nombreuses langues, se compose de 39 pièces, 154 sonnets et quelques poèmes supplémentaires, dont certains ne lui sont pas attribués de manière certaine.

Après des études à Stratford-upon-Avon, dans le Warwickshire, Shakespeare se marie à 18 ans avec Anne Hathaway, avec qui il a trois enfants. À une date inconnue entre 1585 et 1592, il entame sa carrière d'acteur et auteur à succès à Londres au sein des Lord Chamberlain's Men, une troupe dont il est actionnaire. Il semble s'être retiré à Stratford vers 1613 pour y mourir trois ans plus tard. Il ne subsiste guère de traces de l'homme Shakespeare, ce qui a engendré de nombreuses spéculations concernant son apparence physique, sa sexualité, sa religion (en). Des théories marginales avancent que son œuvre a été en réalité écrite par un autre.
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "William_Shakespeare",
      "label": "William Shakespeare",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "William Shakespeare"
        },
        {
          "cle": "birthday",
          "valeur": "26 avril 1564"
        },
        {
          "cle": "age",
          "valeur": "52 ans"
        }
      ]
    },
    {
      "id_entite": "Stratford-upon-Avon",
      "label": "Stratford-upon-Avon",
      "type_entite": "SpatialThing",
      "attributs": []
    },
    {
      "id_entite": "Anne_Hathaway",
      "label": "Anne Hathaway",
      "type_entite": "Person",
      "attributs": []
    },
    {
      "id_entite": "Lord_Chamberlain's_Men",
      "label": "Lord Chamberlain's Men",
      "type_entite": "Organization",
      "attributs": []
    }
  ],
  "relations": [
    {
      "source_id": "William_Shakespeare",
      "type_relation": "

In [ ]:
#avoir definit graphe avant
onto_uri_base = "http://xmlns.com/foaf/0.1/"
envoie_graph_allegro(graphe, conn, onto_uri_base)

Taille avant ajouts: 666


'Succès ! Nouveau nombre de triplets : 676'

In [ ]:
conn.close()


## GraphDb

In [ ]:
!pip install rdflib[graphdb]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 26.5 MB/s eta 0:00:00


In [ ]:
!pip install graphdb-python

Discarding https://files.pythonhosted.org/packages/33/d2/da6ff802df40179111fb110100f7b564829d10daf4f778b799805ad8d97c/GraphDB_Python-0.0.6-py3-none-any.whl (from https://pypi.org/simple/graphdb-python/) (requires-python:>=3.7,<4.0): Requested graphdb-python from https://files.pythonhosted.org/packages/33/d2/da6ff802df40179111fb110100f7b564829d10daf4f778b799805ad8d97c/GraphDB_Python-0.0.6-py3-none-any.whl has inconsistent Name: expected 'graphdb-python', but metadata has 'graphdb python'
Discarding https://files.pythonhosted.org/packages/c6/e6/d28611cd13aa5641840e330eeeb485102b124cc67f12f5bfc5b0f5f713a3/GraphDB_Python-0.0.5-py3-none-any.whl (from https://pypi.org/simple/graphdb-python/) (requires-python:>=3.8,<4.0): Requested graphdb-python from https://files.pythonhosted.org/packages/c6/e6/d28611cd13aa5641840e330eeeb485102b124cc67f12f5bfc5b0f5f713a3/GraphDB_Python-0.0.5-py3-none-any.whl has inconsistent Name: expected 'graphdb-python', but metadata has 'graphdb python'
Discarding https

## Stardog

In [ ]:
!pip install -q pystardog


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 14.7 MB/s eta 0:00:00


In [ ]:
import stardog
from pathlib import Path

mon_utilisateur = userdata.get('Stardog_Username')
mon_mot_de_passe = userdata.get('Stardog_Password')
# specify our endpoint, username, and password
# default values are provided, be sure to change if necessary
conn_details = {
    'endpoint': 'https://sd-4364fc3b.stardog.cloud:5820',
    'username': mon_utilisateur,
    'password': mon_mot_de_passe
}

nom_de_la_base = 'hermes_test'

# Utilisation de stardog.Connection au lieu de stardog.Admin
with stardog.Connection(nom_de_la_base, **conn_details) as conn:
    print(f"Connecté avec succès à la base '{nom_de_la_base}' !")

    # Petite commande pour vérifier que ça fonctionne : afficher la taille du graphe
    print(f"La base contient actuellement {conn.size()} triplets.")

Connecté avec succès à la base 'hermes_test' !
La base contient actuellement 635 triplets.


In [ ]:
def envoie_graph_stardog(graphe, conn, onto_uri_base):
    """Envoie le graphe extrait à Stardog"""
    print(f"Taille avant ajouts: {conn.size()}")

    BASE_URI = "http://mon_projet.org/"
    RDF_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"

    # Au lieu de faire conn.add() à chaque tour, on prépare tous nos triplets en texte
    triplets = ""

    # --- ÉTAPE 1 : LES ENTITÉS ET ATTRIBUTS ---
    for entite_data in graphe.entites:
        # Création de l'URI du sujet (Les < > remplacent createURI)
        sujet = f"<{BASE_URI}{entite_data.id_entite.replace(' ', '_')}>"

        # Définition du type
        nom_classe = entite_data.type_entite.value
        classe_uri = f"<{onto_uri_base}{nom_classe}>"

        # Equivalent de conn.add(sujet, RDF_TYPE, classe_uri)
        triplets += f"{sujet} {RDF_TYPE} {classe_uri} .\n"

        # Ajout des attributs (Datatype Properties)
        for attr in entite_data.attributs:
            propriete = f"<{onto_uri_base}{attr.cle.value}>"

            # Les guillemets autour de la valeur remplacent createLiteral
            valeur = f'"{attr.valeur}"'
            triplets += f"{sujet} {propriete} {valeur} .\n"

    # --- ÉTAPE 2 : LES RELATIONS ---
    for rel in graphe.relations:
        source = f"<{BASE_URI}{rel.source_id.replace(' ', '_')}>"
        type_rel = f"<{onto_uri_base}{rel.type_relation.value}>"
        cible = f"<{BASE_URI}{rel.cible_id.replace(' ', '_')}>"

        # Equivalent de conn.add(source, type_rel, cible)
        triplets += f"{source} {type_rel} {cible} .\n"

    # --- ÉTAPE 3 : ENVOI À STARDOG ---
    # L'équivalent du conn.commit() d'AllegroGraph
    requete_insertion = f"INSERT DATA {{ {triplets} }}"

    # On utilise update() pour envoyer la requête d'écriture
    conn.update(requete_insertion)

    return f"Succès ! Nouveau nombre de triplets : {conn.size()}"

### Envoie a stardog

In [ ]:
texte = """
William Shakespeare est un dramaturge, poète et acteur anglais baptisé/né le 26 avril 1564 à Stratford-upon-Avon et mort le 23 avril 1616 dans la même ville. Surnommé « le Barde d'Avon », « le Barde immortel » ou simplement « le Barde », il est considéré comme l'un des plus grands poètes et dramaturges de langue anglaise. Son œuvre, traduite dans de nombreuses langues, se compose de 39 pièces, 154 sonnets et quelques poèmes supplémentaires, dont certains ne lui sont pas attribués de manière certaine.

Après des études à Stratford-upon-Avon, dans le Warwickshire, Shakespeare se marie à 18 ans avec Anne Hathaway, avec qui il a trois enfants. À une date inconnue entre 1585 et 1592, il entame sa carrière d'acteur et auteur à succès à Londres au sein des Lord Chamberlain's Men, une troupe dont il est actionnaire. Il semble s'être retiré à Stratford vers 1613 pour y mourir trois ans plus tard. Il ne subsiste guère de traces de l'homme Shakespeare, ce qui a engendré de nombreuses spéculations concernant son apparence physique, sa sexualité, sa religion (en). Des théories marginales avancent que son œuvre a été en réalité écrite par un autre.
"""
print("--- Extraction en cours ---")
graphe = extraire_graphe(texte)

print("\n--- RÉSULTAT ---")
print(graphe.model_dump_json(indent=2))

--- Extraction en cours ---

--- RÉSULTAT ---
{
  "entites": [
    {
      "id_entite": "WilliamShakespeare",
      "label": "William Shakespeare",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "William Shakespeare"
        },
        {
          "cle": "birthday",
          "valeur": "26 avril 1564"
        },
        {
          "cle": "age",
          "valeur": "52"
        },
        {
          "cle": "status",
          "valeur": "décédé"
        }
      ]
    },
    {
      "id_entite": "AnneHathaway",
      "label": "Anne Hathaway",
      "type_entite": "Person",
      "attributs": [
        {
          "cle": "name",
          "valeur": "Anne Hathaway"
        }
      ]
    },
    {
      "id_entite": "Stratford-upon-Avon",
      "label": "Stratford-upon-Avon",
      "type_entite": "SpatialThing",
      "attributs": []
    },
    {
      "id_entite": "Londres",
      "label": "Londres",
      "type_entite": "SpatialT

In [ ]:
#avoir definit graphe avant
onto_uri_base = "http://xmlns.com/foaf/0.1/"
envoie_graph_stardog(graphe, conn, onto_uri_base)

Taille avant ajouts: 669


'Succès ! Nouveau nombre de triplets : 682'

# Autre solution

## LLMGraphTransformer

In [ ]:
!pip install -q LLMGraphTransformer
!pip install -q  langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.1 MB/s eta 0:00:00


In [ ]:
from LLMGraphTransformer import LLMGraphTransformer
from LLMGraphTransformer.schema import NodeSchema, RelationshipSchema
from langchain_groq import ChatGroq
from langchain_core.documents import Document


In [ ]:
def lister_attributs_pour_classe(onto, nom_classe):
    classe = onto[nom_classe]
    liste = []
    if not classe:
        return f"La classe '{nom_classe}' n'existe pas dans l'ontologie."

    # On parcourt toutes les DataProperties (Attributs)
    for prop in onto.data_properties():
        # Une propriété est valide si la classe est dans son domaine
        # OU si un des parents de la classe est dans son domaine
        if any(domain_cls in classe.ancestors() for domain_cls in prop.domain):
            liste.append(str(prop.name))
    return liste



def create_relationship_schemas(onto):
    relationship_schemas = []

    for prop in onto.object_properties():
        # On récupère les noms des classes sources (Domain)
        domains = [d.name for d in prop.domain if hasattr(d, 'name')]
        # On récupère les noms des classes cibles (Range)
        ranges = [r.name for r in prop.range if hasattr(r, 'name')]

        if not domains: domains = ["Thing"] # "Thing" est la racine en OWL
        if not ranges: ranges = ["Thing"]

        for d_name in domains:
            for r_name in ranges:
                relationship_schemas.append(
                    RelationshipSchema(d_name,prop.name,r_name
                    )
                )
    return relationship_schemas



def create_node_schema(onto):
  node = []
  for cls in onto.classes():
      node.append(NodeSchema(str(cls), lister_attributs_pour_classe(onto, cls)))
  return node



In [ ]:
lister_attributs_pour_classe(onto, "Document")

['sha1', 'name']

In [ ]:
node_schemas = create_node_schema(onto)
relationship_schemas = create_relationship_schemas(onto)

In [ ]:
text="""

On a dark desert highway
Cool wind in my hair
Warm smell of colitas
Rising up through the air
Up ahead in the distance
I saw a shimmering light
My head grew heavy and my sight grew dim
I had to stop for the night
There she stood in the doorway
I heard the mission bell
And I was thinking to myself
This could be heaven or this could be hell
Then she lit up a candle
And she showed me the way
There were voices down the corridor
I thought I heard them say
Welcome to the Hotel California
Such a lovely place (such a lovely place)
Such a lovely face
Plenty of room at the Hotel California
Any time of year (any time of year)
You can find it here
Her mind is Tiffany-twisted
She got the Mercedes-Benz, uh
She got a lot of pretty, pretty boys
That she calls friends
How they danced in the courtyard
Sweet summer sweat
Some dance to remember
Some dance to forget
So I called up the captain
"Please bring me my wine"
He said, "We haven't had that spirit here since 1969"
And still those voices are calling from far away
Wake you up in the middle of the night
Just to hear them say
Welcome to the Hotel California
Such a lovely place (such a lovely place)
Such a lovely face
They're livin' it up at the Hotel California
What a nice surprise (what a nice surprise)
Bring your alibis
Mirrors on the ceiling
The pink champagne on ice
And she said, "We are all just prisoners here
Of our own device"
And in the master's chambers
They gathered for the feast
They stab it with their steely knives
But they just can't kill the beast
Last thing I remember
I was running for the door
I had to find the passage back
To the place I was before
"Relax, " said the night man
"We are programmed to receive
You can check out any time you like
But you can never leave
"""

document = Document(page_content=text)

In [ ]:
model_name = "llama-3.3-70b-versatile"
llm = ChatGroq(
    model=model_name,
    temperature=0,
)

In [ ]:
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=node_schemas,
    allowed_relationships=relationship_schemas,
    strict_mode = True
)

In [ ]:
graph_document = llm_transformer.convert_to_graph_document(document)

print(f"Nodes: {graph_document.nodes}")
print(f"Relationships: {graph_document.relationships}")

Nodes: [Node(id='Hotel California', type='Foaf.organization', properties={'description': ['a lovely place', 'a lovely face']}), Node(id='Mercedes-Benz', type='Wgs84_pos.spatialthing', properties={'owner': 'She'}), Node(id='She', type='Foaf.person', properties={'mind': 'Tiffany-twisted', 'friends': 'pretty, pretty boys'}), Node(id='Captain', type='Foaf.person', properties={'role': 'captain'}), Node(id='Night Man', type='Foaf.person', properties={'role': 'night man'})]
Relationships: []


In [ ]:
import importlib.metadata